# A next-word machine you can build in a few lines

It counts which word comes straight after each word in a few dozen sentences, then writes by drawing the next word at random, weighted by those counts.

In [ ]:
CORPUS = """
he made a cup of tea for his mother .
she made a cup of coffee for her brother .
he wanted a cup of tea before work .
she wanted a cup of hot water with lemon .
the cat slept in the garden all day .
the dog sat on the mat by the door .
the cat sat on the wall in the sun .
he sat in the kitchen with his tea .
she sat in the garden with a book .
the dog slept by the fire .
he went to the shop in the morning .
she went to the shop for some milk .
he bought milk and bread at the shop .
she bought a cup of coffee on the way .
the shop was busy in the morning .
he read the paper in the kitchen .
she read a book in the garden .
the cat wanted to go out .
the dog wanted to go for a walk .
he took the dog for a walk .
she took the cat to the vet .
he drank his tea in the garden .
she drank her coffee at the table .
he made a cup of tea for the guests .
the kettle was hot so he made tea .
she put the kettle on for a cup of tea .
he left a cup of coffee on the table .
the garden was quiet in the morning .
the kitchen was warm by the fire .
she asked for a cup of tea with milk .
he asked for a cup of water .
the cat sat by the window all morning .
she walked to the shop with the dog .
he walked home in the rain .
it was cold so she made a cup of tea .
the rain stopped in the afternoon .
he cleaned the kitchen after dinner .
she fed the cat in the morning .
"""

## Count which word comes next

In [ ]:
from collections import Counter, defaultdict

tokens = CORPUS.split()
nxt = defaultdict(Counter)
for a, b in zip(tokens, tokens[1:]):
    if a != ".":
        nxt[a][b] += 1

sentences = CORPUS.strip().splitlines()
words = [t for t in tokens if t != "."]
print(len(sentences), "sentences,", len(words), "words")
print('after "of":', dict(nxt["of"]))

38 sentences, 296 words
after "of": {'tea': 6, 'coffee': 3, 'hot': 1, 'water': 1}


## The forecast after "a cup of"

It only looks one word back, so this is the forecast after "of".

In [ ]:
def forecast(word):
    c = nxt[word]
    total = sum(c.values())
    return {w: n / total for w, n in c.most_common()}

for w, p in forecast("of").items():
    print(f"{w:7} {p:.0%}")

tea     55%
coffee  27%
hot     9%
water   9%


## Let it write

Draw the next word from the counts, again and again, until a full stop.

In [ ]:
import random

def write(start, seed):
    rng = random.Random(seed)
    out = [start]
    while out[-1] != "." and len(out) < 25:
        options, counts = zip(*nxt[out[-1]].items())
        out.append(rng.choices(options, weights=counts)[0])
    return " ".join(out)

for seed in range(5):
    print(write("he", seed))

he asked for a cup of tea in the kitchen was cold so he left a cup of water .
he made tea .
he cleaned the cat slept by the morning .
he wanted to the shop was busy in the dog slept by the table .
he wanted a cup of tea .


## Always the tallest bar

In [ ]:
out = ["he"]
while out[-1] != ".":
    out.append(nxt[out[-1]].most_common(1)[0][0])
print(" ".join(out))

he made a cup of tea .


## Temperature

Raise each chance to the power 1/T and rescale so they add up to 1. Low T: the favourite wins more. High T: more even.

In [ ]:
def with_temperature(probs, t):
    w = {k: p ** (1 / t) for k, p in probs.items()}
    z = sum(w.values())
    return {k: v / z for k, v in w.items()}

for t in (0.3, 1.0, 2.5):
    print(t, {k: f"{v:.0%}" for k, v in with_temperature(forecast("of"), t).items()})

0.3 {'tea': '91%', 'coffee': '9%', 'hot': '0%', 'water': '0%'}
1.0 {'tea': '55%', 'coffee': '27%', 'hot': '9%', 'water': '9%'}
2.5 {'tea': '37%', 'coffee': '28%', 'hot': '18%', 'water': '18%'}


## Surprise (the loss)

The surprise when the right word had chance p is minus the log of p. A small open model, after "... 2. Use fresh", gave "water" 69.52% and "and" 4.96%.

In [ ]:
import math

for word, p in [("water", 0.6952), ("and", 0.0496)]:
    print(f"{word}: {p:.0%} -> surprise {-math.log(p):.2f}")

water: 70% -> surprise 0.36
and: 5% -> surprise 3.00


## Try your own

Add your own sentences to CORPUS above (spaces around the full stop), run all the cells again, and change the starting word here.

In [ ]:
print({w: f"{p:.0%}" for w, p in list(forecast("the").items())[:6]})
print(write("she", 7))

{'cat': '13%', 'garden': '11%', 'dog': '11%', 'shop': '11%', 'kitchen': '9%', 'morning': '9%'}
she went to the cat wanted a cup of tea .
